# CASMI 2026 — 4-channel retrieval + learned reranker

Four evidence channels over a 275,810-structure candidate pool, fused by a
learned reranker:

1. **Channel 1** — direct spectral match against `train.parquet` using spectral
   entropy similarity (Li et al. 2021), aggregated per molecule across its spectra.
2. **Channel 2** — mass-shifted analog propagation: search a ±200 Da window with
   shift-aware similarity, then propagate structural evidence by fingerprint
   Tanimoto to the matched analogs.
3. **Channel 4** — FPNet, a spectrum→fingerprint transformer trained on the real
   train set (Azure, val cosine similarity 0.71), scored per candidate as an exact
   Bayes log-likelihood dot product against the predicted fingerprint logits.
4. **Channel 5** — MetFrag-lite in-silico fragmentation: enumerate 1-2 bond-break
   fragments per candidate and score how much observed MS2 intensity they explain.

All four channels' per-candidate features feed a bagged
`HistGradientBoostingClassifier` ensemble (2 class-1 priors x 4 seeds), trained
offline against our own held-out split — see `docs/06-implementation-plan.md`
and `.kiro/memory/` for the honest (non-leaked) MRR@25 numbers this earned.

**On the public leaderboard score:** the visible `test.parquet` is a sample of
`train.parquet`, so Channel 1 finds a perfect match for ~100% of molecules. The
diagnostics at the end quantify this. A high public score here mostly measures
whether a library lookup was implemented, not generalisation to the hidden set —
our own held-out split is the number that actually matters.

In [ ]:
# No internet in a scored run, so RDKit comes from a pre-attached wheel.
# Kaggle's image has numpy/pandas/pyarrow/numba/sklearn/torch but NOT rdkit
# (verified by probe) — torch and sklearn are already present, so only RDKit
# needs installing and only the FPNet/ranker weights need attaching.
import glob, subprocess, sys, os, time

T0 = time.time()

print('/kaggle/input contains:', sorted(os.listdir('/kaggle/input')))

# Search the whole input tree rather than assuming a mount path, so a renamed
# or re-versioned dataset does not silently skip the RDKit install.
wheels = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
print('rdkit wheels found:', wheels)
if not wheels:
    raise RuntimeError('RDKit wheel not found under /kaggle/input; attach the assets dataset')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-index', wheels[0]],
    check=True,
)
print('installed', os.path.basename(wheels[0]))

# Our package ships in the same dataset; locate it by its __init__.py.
inits = glob.glob('/kaggle/input/**/casmi/__init__.py', recursive=True)
if not inits:
    raise RuntimeError('casmi package not found under /kaggle/input')
pkg_parent = os.path.dirname(os.path.dirname(inits[0]))
sys.path.insert(0, pkg_parent)
print('casmi source:', pkg_parent)

import rdkit
from casmi import __version__
print('rdkit', rdkit.__version__, '| casmi', __version__)

In [ ]:
import numpy as np

from casmi.candidates.pool import CandidatePool
from casmi.data.loaders import load_query_molecules, load_spectral_library
from casmi.pipeline import run_pipeline, write_submission

COMP = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
if not os.path.isdir(COMP):
    hits = glob.glob('/kaggle/input/**/test.parquet', recursive=True)
    COMP = os.path.dirname(hits[0])
print('competition data:', COMP)

pool_path = glob.glob('/kaggle/input/**/pool.npz', recursive=True)[0]
pool = CandidatePool.load(pool_path)
print(f'pool: {len(pool):,} structures, {pool.n_bits} fingerprint bits')

In [ ]:
# Channel 4: FPNet checkpoint. Optional — the pipeline degrades cleanly to
# Channels 1/2/5 if this (or torch itself) is unavailable, so a missing
# checkpoint is a warning, not a hard failure.
fpnet_model, fpnet_config = None, None
fpnet_paths = glob.glob('/kaggle/input/**/fpnet_final.pt', recursive=True)
if fpnet_paths:
    try:
        from casmi.models.train import load_checkpoint
        fpnet_model, fpnet_config = load_checkpoint(fpnet_paths[0], device='cpu')
        print(f'loaded FPNet checkpoint: {fpnet_paths[0]} ({fpnet_config.n_bits} bits)')
    except Exception as exc:
        print(f'WARNING: failed to load FPNet checkpoint, skipping Channel 4: {exc}')
else:
    print('no FPNet checkpoint found under /kaggle/input; Channel 4 disabled')

# The learned reranker replacing weighted fusion. Same fallback contract:
# missing artifact -> transparent weighted fusion, not a hard failure.
reranker = None
ranker_paths = glob.glob('/kaggle/input/**/ranker.pkl', recursive=True)
if ranker_paths:
    try:
        from casmi.channels.ranker import Reranker
        reranker = Reranker.load(ranker_paths[0])
        print(f'loaded reranker: {ranker_paths[0]} ({len(reranker.models)} models)')
    except Exception as exc:
        print(f'WARNING: failed to load reranker, falling back to weighted fusion: {exc}')
else:
    print('no reranker found under /kaggle/input; using weighted fusion')

In [ ]:
# The full train set is the reference library for both channels.
library = load_spectral_library(f'{COMP}/train.parquet')
print(f'library: {library.n_spectra:,} spectra, {library.n_finite:,} with resolvable mass')

molecules = load_query_molecules(f'{COMP}/test.parquet')
print(f'test: {len(molecules):,} molecules, {sum(m.n_spectra for m in molecules):,} spectra')
print(f'setup took {time.time() - T0:.0f}s')

In [ ]:
# Kaggle gives 4 cores, so this is markedly slower than a large box; budget is 9h.
# Channel 5 (fragmentation) adds one RDKit fragment enumeration per candidate,
# so it is the most expensive addition — still comfortably within budget at
# ~80 candidates/molecule x 400 molecules (see docs/06-implementation-plan.md).
result = run_pipeline(
    molecules,
    library,
    pool,
    progress_every=25,
    fragmentation_channel=True,
    fpnet_model=fpnet_model,
    fpnet_config=fpnet_config,
    fpnet_device='cpu',
    reranker=reranker,
)
print(f'inference done at {time.time() - T0:.0f}s')

In [ ]:
import csv

with open(f'{COMP}/sample_submission.csv', newline='') as handle:
    sample_ids = [row['molecule_id'] for row in csv.DictReader(handle)]

write_submission(result.predictions, sample_ids, 'submission.csv')

# Validate strictly: a malformed submission scores zero.
with open('submission.csv', newline='') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == len(sample_ids), f'{len(rows)} rows vs {len(sample_ids)} expected'
assert [r['molecule_id'] for r in rows] == sample_ids, 'molecule_id order mismatch'
assert all(len(r['smiles'].split(';')) == 25 for r in rows), 'every row needs 25 candidates'
assert not any(s == '' for r in rows for s in r['smiles'].split(';')), 'empty SMILES present'
print(f'submission.csv OK: {len(rows)} rows x 25 candidates')

In [ ]:
# Leakage + channel-usage diagnostics — the reason this public score should
# not be trusted on its own; see docs/05-community-intel.md.
lib = np.array([d.best_library_similarity for d in result.diagnostics])
ana = np.array([d.best_analog_similarity for d in result.diagnostics])
frag = np.array([d.best_fragmentation_score for d in result.diagnostics])
fpnet_scores = np.array([d.best_fpnet_score for d in result.diagnostics])
reranked_fraction = np.mean([d.used_reranker for d in result.diagnostics])
contribution = result.channel_contribution()

print(f'best_library_sim       mean {lib.mean():.4f}  min {lib.min():.4f}  max {lib.max():.4f}')
print(f'best_analog_sim        mean {ana.mean():.4f}')
print(f'best_fragmentation_sc  mean {frag.mean():.4f}')
print(f'best_fpnet_score       mean {fpnet_scores.mean():.4f}')
print(f'used_reranker          {reranked_fraction:.1%} of molecules')
print(f'perfect library match for {(lib >= 0.9999).mean():.1%} of molecules')
print(f'carried by library        {contribution["library"]:.1%}')
print(f'carried by analog/other   {contribution["analog_or_other"]:.1%}')

if (lib >= 0.9999).mean() > 0.5:
    print(
        '\nNOTE: most molecules have an exact library match. The visible test set is\n'
        'a sample of train, so this score largely reflects that leak rather than\n'
        'generalisation. Our held-out-split MRR@25 is the trustworthy number.'
    )
print(f'\ntotal runtime {time.time() - T0:.0f}s')